# BPR ベース + 2-hop 実験

**ベース**: `submission_atmacup_implicit_bpr16.csv`（Public 0.76101）と同じ BPR 16 埋め込み。
ここに **2-hop 集約**（その映画をレビューした批評家たちの Fresh 率平均・critic_te 平均・人数など）を 1 列ずつ追加し、スコアの増減を確認する。

- ロジックは `lib/two_hop.py` に実装。ノートブックから `run_experiment` を呼んで実行する。
- 提出ファイルは `outputs/submissions/submission_2hop_{experiment_name}.csv`。

## 1. セットアップ

In [ ]:
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path
from lib.improvement_candidates import get_setup
from lib.two_hop import (
    run_experiment,
    TWO_HOP_COLUMNS,
    TWO_HOP_FRESH_MEAN,
    TWO_HOP_CRITIC_TE_MEAN,
    TWO_HOP_REVIEW_COUNT,
)

ctx = get_setup(seed=42, output_dir="outputs", use_best_pipeline=True)

print(f"Train: {len(ctx.train):,}, Test: {len(ctx.test):,}, Features: {len(ctx.features)}")
print(f"提出先: {ctx.submissions_dir}")
print(f"2-hop 列: {TWO_HOP_COLUMNS}")

## 2. 実験実行（BPR のみ → 2-hop を 1 列ずつ追加）

下のセルで実験を並べている。1 本ずつ実行して Public スコアを確認し、効いた列を残して次を足していく。

In [ ]:
def _report(name: str, r: dict):
    if r.get("path"):
        p = r["path"]
        name_part = p.name if hasattr(p, "name") else Path(p).name
        print(f"  [{name}] → {name_part}  ({'OK' if r.get('ok') else r.get('message', '')})")
    else:
        print(f"  [{name}] スキップ: {r.get('message', '')}")

results = []

# 0: BPR のみ（ベースライン 0.76101 の再現）
r0 = run_experiment(ctx, "bpr_only", use_2hop_cols=None)
_report("0 BPR のみ", r0)
results.append(("0 BPR のみ", r0))

# 1: BPR + 2-hop 1 列（Fresh 率平均）
r1 = run_experiment(ctx, "bpr_fresh", use_2hop_cols=[TWO_HOP_FRESH_MEAN])
_report("1 BPR + movie_fresh_rate_mean", r1)
results.append(("1 BPR + fresh_mean", r1))

# 2: BPR + 2-hop 1 列（critic_te 平均）
r2 = run_experiment(ctx, "bpr_critic_te", use_2hop_cols=[TWO_HOP_CRITIC_TE_MEAN])
_report("2 BPR + movie_critic_te_mean", r2)
results.append(("2 BPR + critic_te_mean", r2))

# 3: BPR + 2-hop 1 列（レビュー数）
r3 = run_experiment(ctx, "bpr_count", use_2hop_cols=[TWO_HOP_REVIEW_COUNT])
_report("3 BPR + movie_review_count", r3)
results.append(("3 BPR + review_count", r3))

# 4: BPR + 2-hop 全列
r4 = run_experiment(ctx, "bpr_all2hop", use_2hop_cols=TWO_HOP_COLUMNS)
_report("4 BPR + 全2-hop", r4)
results.append(("4 BPR + all2hop", r4))

print(f"\n→ {sum(1 for _, r in results if r.get('ok'))} / {len(results)} 本 OK")
print("提出後、Public スコアをメモして効いた列を決める。")

## 3. 1 本だけ試す（実験名・2-hop 列を変えて実行）

上のセルで全パターン回す代わりに、ここで 1 本だけ `run_experiment` を呼ぶ。`experiment_name` と `use_2hop_cols` を変えて追加実験する。

In [ ]:
# 例: BPR + fresh_mean と count の 2 列だけ
# r = run_experiment(ctx, "bpr_fresh_count", use_2hop_cols=[TWO_HOP_FRESH_MEAN, TWO_HOP_REVIEW_COUNT])
# _report("BPR + fresh_count", r)

# 1 本だけ実行するときは下のコメントを外して experiment_name と use_2hop_cols を編集
# r = run_experiment(ctx, "my_experiment", use_2hop_cols=[TWO_HOP_FRESH_MEAN])
# _report("my_experiment", r)

## 4. 提出ファイル一覧

In [ ]:
for p in sorted(ctx.submissions_dir.glob("submission_2hop_*.csv")):
    print(p.name)